# Ensemble Retriever

El `EnsembleRetriever` toma una lista de buscadores (`retrievers`) como entrada y agrupa (`ensemble`) los resultados de sus métodos `get_relevant_documents()`, para después reorganizar los resultados usando el algoritmo de Fusión de Rango Recíproco (`Reciprocal Rank Fusion`).

Al aprovechar las fortalezas de diferentes algoritmos, el `EnsembleRetriever` puede lograr un rendimiento mejor que cualquier algoritmo individual.

Un patrón común es combinar un buscador disperso (como BM25) con un buscador denso (como similaridad de incrustación/embedding), ya que sus fortalezas son complementarias. Esto también se conoce como "búsqueda híbrida".

- **Buscador Disperso (Sparse Retriever)**: Es eficaz para encontrar documentos relevantes basados en palabras clave.
- **Buscador Denso (Dense Retriever)**: Es eficaz para encontrar documentos relevantes basados en similitud semántica.

## Librerías

In [1]:
from dotenv import load_dotenv
from langchain_mistralai import MistralAIEmbeddings
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever #BM25Retriever es un buscador disperso
from langchain_chroma import Chroma

from src.langchain_docs_loader import load_langchain_docs_splitted

load_dotenv()

True

In [4]:
%%capture
!pip install rank_bm25

## Carga de datos

In [2]:
docs = load_langchain_docs_splitted()
len(docs)

556

## Inicialización de retrievers independientes

In [5]:
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

vector_retriever = Chroma.from_documents(
    docs, embedding=MistralAIEmbeddings()
).as_retriever(search_kwargs={"k": 2})

In [6]:
bm25_retriever.invoke("Como utilizar un retriever con langchain expresion language?")

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain', 'language': 'en'}, page_content='## \u200bOverview\n\nIn this tutorial we will build a [retrieval](/oss/python/langchain/retrieval) agent using LangGraph.\n\nLangChain offers built-in [agent](/oss/python/langchain/agents) implementations, implemented using [LangGraph](/oss/python/langgraph/overview) primitives. If deeper customization is required, agents can be implemented directly in LangGraph. This guide demonstrates an example implementation of a retrieval agent. [Retrieval](/oss/python/langchain/retrieval) agents are useful when you want an LLM to make a decision about whether to retrieve context from a vectorstore or respond to the user directly.\n\nBy the end of the tutorial we will have done the following:\n\n1. Fetch and preprocess documents that will be used for retrieval.\n2. Index those documents for semantic sea

In [7]:
vector_retriever.invoke("Como utilizar un retriever con langchain expresion language?")

[Document(id='18b89d82-5dc7-4e44-aece-12cc674c06b4', metadata={'source': 'https://docs.langchain.com/oss/python/integrations/retrievers', 'language': 'en', 'title': 'Retrievers - Docs by LangChain'}, page_content='A [retriever](/oss/python/langchain/retrieval#building-blocks) is an interface that returns documents given an unstructured query.\nIt is more general than a vector store.\nA retriever does not need to be able to store documents, only to return (or retrieve) them.\nRetrievers can be created from vector stores, but are also broad enough to include [Wikipedia search](/oss/python/integrations/retrievers/wikipedia) and [Amazon Kendra](/oss/python/integrations/retrievers/amazon_kendra_retriever).\n\nRetrievers accept a string query as input and return a list of [Document](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) objects as output.\n\nNote that all [vector stores](/oss/python/integrations/vectorstores) can be cast to r

## Ensamblaje de retrievers

In [8]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

In [9]:
ensemble_retriever.invoke(
    "¿Cómo utilizar un retriever con langchain expression language?"
)

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain', 'language': 'en'}, page_content='## \u200bOverview\n\nIn this tutorial we will build a [retrieval](/oss/python/langchain/retrieval) agent using LangGraph.\n\nLangChain offers built-in [agent](/oss/python/langchain/agents) implementations, implemented using [LangGraph](/oss/python/langgraph/overview) primitives. If deeper customization is required, agents can be implemented directly in LangGraph. This guide demonstrates an example implementation of a retrieval agent. [Retrieval](/oss/python/langchain/retrieval) agents are useful when you want an LLM to make a decision about whether to retrieve context from a vectorstore or respond to the user directly.\n\nBy the end of the tutorial we will have done the following:\n\n1. Fetch and preprocess documents that will be used for retrieval.\n2. Index those documents for semantic sea